# NPS Driver Analysis

### Customer Experience Churn Analytics — Banking-AI

This notebook explains how banks analyze survey data to understand what makes customers happy (Promoters) or unhappy (Detractors):

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of customer experience and churn analytics terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the customer experience and churn analytics prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Customer churn prediction, segmentation, lifetime value, and retention strategy in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** The Net Promoter Score (NPS) asks one simple question: "How likely are you to recommend us to a friend?" (0-10). However, just knowing the score isn't enough. Banks need to know *why* the score is high or low. Is it because the mobile app is hard to use? Or because the interest rates are too low? Driver analysis helps prioritize where to invest money to improve the customer experience.

**Input data used:** Survey ratings (1-10) for App Ease of Use, Branch Wait Time, Interest Rates, and Staff Friendliness.

**What we predict:** The likelihood to recommend (`NPS Score`).

**ML method used:** Linear Regression (for Feature Importance).

**Learning goal:** Learn how to use regression coefficients to explain the "why" behind a target metric.

**Primary stakeholders:** relationship managers, retention teams, marketing analysts, and branch managers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic NPS Survey Dataset

We simulate 2,000 customer survey responses.

In [ ]:
n = 2000
rng = RNG

app_ease = rng.integers(1, 11, n)
wait_time_rating = rng.integers(1, 11, n)
interest_rates_rating = rng.integers(1, 11, n)
staff_friendliness = rng.integers(1, 11, n)

# Logic for NPS Score: 
# We assume App Ease and Interest Rates are the biggest drivers for this bank.
nps_score = (
    0.4 * app_ease +
    0.1 * wait_time_rating +
    0.3 * interest_rates_rating +
    0.2 * staff_friendliness +
    rng.normal(0, 0.5, n)
).clip(0, 10).round().astype(int)

df = pd.DataFrame({
    'app_ease': app_ease,
    'wait_time': wait_time_rating,
    'interest_rates': interest_rates_rating,
    'staff_friendliness': staff_friendliness,
    'nps_score': nps_score
})

# NPS Category Logic
def categorize_nps(score):
    if score >= 9: return 'Promoter'
    elif score >= 7: return 'Passive'
    else: return 'Detractor'

df['category'] = df['nps_score'].apply(categorize_nps)

print(df['category'].value_counts())
df.head()

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

The official formula: `% Promoters - % Detractors`.

In [ ]:
counts = df['category'].value_counts(normalize=True) * 100
nps_final = counts.get('Promoter', 0) - counts.get('Detractor', 0)
print(f"The Bank's overall Net Promoter Score (NPS) is: {nps_final:.1f}")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
baseline_value = y.mean()
print(f'Baseline: predict mean = {baseline_value:.4f}')

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
X = df[['app_ease', 'wait_time', 'interest_rates', 'staff_friendliness']]
y = df['nps_score']

model = LinearRegression()
model.fit(X, y)

drivers = pd.Series(model.coef_, index=X.columns)
print("Regression Coefficients (Importance):")
print(drivers.sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(8, 5))
drivers.sort_values().plot(kind='barh', color='skyblue')
plt.title('NPS Drivers: What Impacts Recommendation Likelyhood?')
plt.xlabel('Impact Strength (Coefficient)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "NPS Drivers: What Impacts Recommendation Likelyhood?" with Impact Strength (Coefficient) on the x-axis. The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- **App Ease of Use** is the #1 driver of customer recommendation for this bank.
- **Interest Rates** also play a major role.
- **Wait Time** has the lowest impact, meaning customers are willing to wait a bit if the app works perfectly and the rates are competitive.

**Banking Context:** Based on this data, the bank should prioritize its IT budget on mobile app improvements rather than hiring more branch staff to reduce wait times, as app quality is twice as important to their customers' loyalty.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
print("Scenario analysis: inspect cluster profiles and map them to actionable banking segments.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end customer experience and churn analytics workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Churn models that rely on demographic features risk discriminatory retention offers.
- Aggressive retention tactics driven by model outputs may erode customer trust.
- Customers should benefit from personalisation, not be manipulated by it.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in customer experience and churn analytics?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.